In [0]:
from pyspark.sql.functions import col

silver_schema = dbutils.widgets.get("silver_schema")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

# load cleaned customers from silver
customers_df = spark.table(f"{silver_schema}.customers_cleaned")

lookup_path = "/Volumes/workspace/default/lookup"

city_state_df = spark.read.format("json").load(lookup_path)

customers_enriched = customers_df.join(
    city_state_df,
    customers_df.city == city_state_df.city,
    "left"
).drop(city_state_df.city) 

customers_enriched.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schema}.customers_enriched")